# 基于Agent轨迹的SFT：前期技术验证

## 1. 下载原始Agent轨迹数据集

In [1]:
from pathlib import Path

# 下载数据集
def download_dataset(data_dir: str, dataset_id: str) -> Path:
    raw_dir = Path(data_dir) / "raw"
    if raw_dir.exists():
        return raw_dir

    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=dataset_id,
        repo_type="dataset",
        allow_patterns=["*.jsonl"],
        local_dir=str(raw_dir),
    )
    return raw_dir

# 下载真实pi-mono轨迹
data_dir = "./sft-data"
dataset_id = "badlogicgames/pi-mono"
raw_dir = download_dataset(data_dir, dataset_id)
raw_files = sorted(raw_dir.glob("*.jsonl"))
if not raw_files:
    raise FileNotFoundError(f"真实轨迹目录内没有 JSONL：{raw_dir}")
print("原始文件目录:", raw_dir)
print("原始文件数量:", len(raw_files), "第一个文件:", raw_files[0].name)
# 查看第一个文件
with raw_files[0].open(encoding="utf-8", errors="replace") as handle:
    print(handle.read())

原始文件目录: sft-data/raw
原始文件数量: 627 第一个文件: 2026-01-16T02-31-35-233Z_8072a61b-67e6-4618-8f35-5c5616aea2be.jsonl
{"type":"session","version":3,"id":"8072a61b-67e6-4618-8f35-5c5616aea2be","timestamp":"2026-01-16T02:31:35.233Z","cwd":"/Users/badlogic/workspaces/pi-mono"}
{"type":"model_change","id":"492a4781","parentId":null,"timestamp":"2026-01-16T02:31:35.245Z","provider":"anthropic","modelId":"claude-opus-4-5"}
{"type":"thinking_level_change","id":"295a987c","parentId":"492a4781","timestamp":"2026-01-16T02:31:35.245Z","thinkingLevel":"high"}
{"type":"message","id":"b95c7ac4","parentId":"295a987c","timestamp":"2026-01-16T02:31:43.903Z","message":{"role":"user","content":[{"type":"text","text":"hello"}],"timestamp":1768530703898}}
{"type":"message","id":"0faa9ad5","parentId":"b95c7ac4","timestamp":"2026-01-16T02:31:49.511Z","message":{"role":"assistant","content":[{"type":"thinking","thinking":"The user said hello. I should greet them back and ask what they'd like to work on. According to th

## 2. 理解原始Agent轨迹的数据结构

In [2]:
import json
from collections import Counter

# 读取单个Session的全部事件
def load_session(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

# 寻找包含工具调用的 Session
def find_tool_session(files):
    for path in files:
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                if '"type":"toolCall"' in line:
                    return path
    raise ValueError("没有找到包含工具调用的 Session")

sample_file = find_tool_session(raw_files)
events = load_session(sample_file)

print("Session 文件:", sample_file.name)
print("事件总数:", len(events))

Session 文件: 2026-01-16T02-37-34-075Z_4293a326-81ca-4327-b450-85275e1ca645.jsonl
事件总数: 325


In [3]:
# 统计顶层事件类型
event_types = Counter(
    event.get("type") for event in events
)

# 提取 message 类型的事件
message_events = [
    event for event in events
    if event.get("type") == "message"
]

# 统计对话角色
message_roles = Counter(
    event["message"].get("role")
    for event in message_events
)

print("事件类型：", event_types)
print("消息角色：", message_roles)

事件类型： Counter({'message': 320, 'session_info': 2, 'session': 1, 'model_change': 1, 'thinking_level_change': 1})
消息角色： Counter({'assistant': 157, 'toolResult': 144, 'user': 19})


In [4]:
content_types = Counter()

# 统计不同内容片段的类型
for event in message_events:
    content = event["message"].get("content", [])

    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict):
                content_types[block.get("type")] += 1

print("Content 类型统计：", content_types)

Content 类型统计： Counter({'text': 213, 'thinking': 156, 'toolCall': 144, 'image': 1})


In [5]:
# 查看第一条 Assistant 消息的结构
assistant_events = [
    event for event in message_events
    if event["message"].get("role") == "assistant"
]

sample = assistant_events[0]

print("事件 ID:", sample.get("id"))
print("父事件 ID:", sample.get("parentId"))
print("消息角色:", sample["message"]["role"])

for block in sample["message"].get("content", []):
    print("Content 类型:", block.get("type"))
print(sample)

事件 ID: c6794344
父事件 ID: 9114fdd0
消息角色: assistant
Content 类型: thinking
Content 类型: toolCall
{'type': 'message', 'id': 'c6794344', 'parentId': '9114fdd0', 'timestamp': '2026-01-16T02:38:02.488Z', 'message': {'role': 'assistant', 'content': [{'type': 'thinking', 'thinking': 'The user wants me to analyze GitHub issue #732. Let me fetch the issue details first.', 'thinkingSignature': 'Ev0BCkYICxgCKkAJNWVZQPvtMWtCEn3U1kENAwCYv2K6upjZmjmi5sVaPkqN7pWYsBnrOnjWEJ4E1f2NHecoM7GoIYw1bjmLHk9sEgxsBBEEYwPZdc4JeekaDMZFWRM4C3HE0z1NoyIwq97UsedAZA8wIcMbseNr5aB5m1h3So0KiN0bbPLR3s+YYQJN3+o7m9CO2srFjC+qKmXHkxV5xKNhIGg8zaUj3bLrTCxTMKf5cY4UQmvSK7+I306kKwiRZ9w5DwqYH+Mf10zNV2fMxTDcp3sn1hlPA53cKDRd2ZI9zZ3M7GWSW+Stk3xzH6Gtq5qxl5wsKJE9/E7/hoy28xgB'}, {'type': 'toolCall', 'id': 'toolu_01HK4MwpXYrA79PdGFxamC1S', 'name': 'bash', 'arguments': {'command': 'gh issue view 732 --repo badlogic/pi-mono --json title,body,comments,labels,state'}}], 'api': 'anthropic-messages', 'provider': 'anthropic', 'model': 'claude-opus-4-5

In [6]:
# 提取第一条包含工具调用的 Assistant 消息
assistant_event = next(
    event for event in message_events
    if event["message"]["role"] == "assistant"
    and any(
        block.get("type") == "toolCall"
        for block in event["message"].get("content", [])
    )
)

# 提取工具调用
tool_call = next(
    block for block in assistant_event["message"]["content"]
    if block.get("type") == "toolCall"
)

# 根据 toolCallId 查找对应的工具执行结果
tool_result = next(
    event for event in message_events
    if event["message"]["role"] == "toolResult"
    and event["message"].get("toolCallId") == tool_call["id"]
)

print("工具名称:", tool_call["name"])
print("调用参数:", tool_call["arguments"])
print("工具调用 ID:", tool_call["id"])
print("工具结果 ID:", tool_result["message"]["toolCallId"])
print("执行结果:", str(tool_result["message"].get("content"))[:1000])

工具名称: bash
调用参数: {'command': 'gh issue view 732 --repo badlogic/pi-mono --json title,body,comments,labels,state'}
工具调用 ID: toolu_01HK4MwpXYrA79PdGFxamC1S
工具结果 ID: toolu_01HK4MwpXYrA79PdGFxamC1S
执行结果: [{'type': 'text', 'text': '{"body":"When you load a prompt with `setEditorText` in that is larger than the screen, then everything falls apart:\\n\\n<img width=\\"2058\\" height=\\"2144\\" alt=\\"Image\\" src=\\"https://github.com/user-attachments/assets/19a877b1-07e8-4a8a-94fe-ecf6558ec469\\" />\\n\\nEsc also stops working (ctrl+c still works though).  I guess the editor component would have to start scrolling if it\'s reaching the limits of the terminal.","comments":[],"labels":[],"state":"OPEN","title":"Prompts larger than the screen fuck up everything"}\n'}]


In [7]:
from collections import defaultdict

# 建立事件 ID 到事件对象的映射
event_map = {
    event["id"]: event
    for event in events
    if "id" in event
}

# 建立父事件到子事件的映射
children = defaultdict(list)

for event in events:
    parent_id = event.get("parentId")

    if parent_id is not None:
        children[parent_id].append(event["id"])

# 查找存在多个子事件的节点，判断会话是否存在分支
branch_nodes = {
    parent_id: child_ids
    for parent_id, child_ids in children.items()
    if len(child_ids) > 1
}

print("事件节点数:", len(event_map))
print("分支节点数:", len(branch_nodes))

for parent_id, child_ids in list(branch_nodes.items())[:5]:
    print(f"父节点 {parent_id} -> 子节点 {child_ids}")

事件节点数: 325
分支节点数: 0


## 3. 将原始Agent消息转换成标准对话格式

In [8]:
# 提取文本内容，暂不处理thinking推理部分
def extract_text_content(content):
    if isinstance(content, str):
        return content.strip()

    if not isinstance(content, list):
        return ""

    texts = []

    for block in content:
        if not isinstance(block, dict):
            continue

        if block.get("type") == "text":
            texts.append(block.get("text", ""))

        elif block.get("type") == "image":
            texts.append("[image omitted]")

    return "\n".join(texts).strip()

In [9]:
# 转换工具调用
def convert_tool_call(block):
    if not block.get("name") or not block.get("id"):
        return None

    return {
        "id": block["id"],
        "type": "function",
        "function": {
            "name": block["name"],
            "arguments": block.get("arguments") or {}
        }
    }

In [10]:
# 将原始User/Assistant/Tool消息统一转换成标注对话格式
def convert_event_to_message(event):
    # 跳过非消息事件
    if event.get("type") != "message":
        return None

    raw_message = event.get("message") or {}
    role = raw_message.get("role")
    content = raw_message.get("content", [])

    # User 消息
    if role == "user":
        text = extract_text_content(content)
        if not text:
            return None

        return {
            "role": "user",
            "content": text
        }

    # Assistant 消息
    elif role == "assistant":
        text = extract_text_content(content)

        tool_calls = []
        for block in content if isinstance(content, list) else []:
            if isinstance(block, dict) and block.get("type") == "toolCall":
                call = convert_tool_call(block)
                if call is not None:
                    tool_calls.append(call)

        if not text and not tool_calls:
            return None

        message = {
            "role": "assistant",
            "content": text
        }

        if tool_calls:
            message["tool_calls"] = tool_calls

        return message

    # Tool Result 消息
    elif role == "toolResult":
        tool_call_id = raw_message.get("toolCallId")
        if not tool_call_id:
            return None

        return {
            "role": "tool",
            "tool_call_id": tool_call_id,
            "name": raw_message.get("toolName", "unknown"),
            "content": extract_text_content(content) or "[empty tool result]"
        }

    return None

In [11]:
from collections import Counter
from pprint import pprint

# 转换当前 Session 中的所有消息
messages = []

for event in events:
    message = convert_event_to_message(event)

    if message is not None:
        messages.append(message)

print("转换后的消息总数:", len(messages))
print("角色统计:", Counter(msg["role"] for msg in messages))

# 查看第一条包含工具调用的 Assistant 消息
sample_assistant = next(
    msg for msg in messages
    if msg["role"] == "assistant" and msg.get("tool_calls")
)

pprint(sample_assistant)

转换后的消息总数: 319
角色统计: Counter({'assistant': 156, 'tool': 144, 'user': 19})
{'content': '',
 'role': 'assistant',
 'tool_calls': [{'function': {'arguments': {'command': 'gh issue view 732 '
                                                       '--repo '
                                                       'badlogic/pi-mono '
                                                       '--json '
                                                       'title,body,comments,labels,state'},
                              'name': 'bash'},
                 'id': 'toolu_01HK4MwpXYrA79PdGFxamC1S',
                 'type': 'function'}]}


In [12]:
# 原始assistant消息总数为157，转换后为156，查看被过滤的消息
skipped_assistants = [
    event for event in message_events
    if event["message"].get("role") == "assistant"
    and convert_event_to_message(event) is None
]

for event in skipped_assistants:
    print("事件 ID:", event["id"])
    print("Content 类型:", [
        block.get("type")
        for block in event["message"].get("content", [])
        if isinstance(block, dict)
    ])
print(skipped_assistants)

事件 ID: 9cd758be
Content 类型: []
[{'type': 'message', 'id': '9cd758be', 'parentId': 'ecc3471b', 'timestamp': '2026-01-16T03:09:48.457Z', 'message': {'role': 'assistant', 'content': [], 'api': 'anthropic-messages', 'provider': 'anthropic', 'model': 'claude-opus-4-5', 'usage': {'input': 0, 'output': 0, 'cacheRead': 0, 'cacheWrite': 0, 'totalTokens': 0, 'cost': {'input': 0, 'output': 0, 'cacheRead': 0, 'cacheWrite': 0, 'total': 0}}, 'stopReason': 'aborted', 'timestamp': 1768532986342, 'errorMessage': 'Operation aborted'}}]


In [13]:
# 检查所有工具调用ID是否匹配
call_ids = {
    call["id"]
    for msg in messages
    for call in msg.get("tool_calls", [])
}

result_ids = {
    msg["tool_call_id"]
    for msg in messages
    if msg["role"] == "tool"
}

print("工具调用数:", len(call_ids))
print("工具返回数:", len(result_ids))
print("没有返回结果的调用数:", len(call_ids - result_ids))
print("没有对应调用的返回数:", len(result_ids - call_ids))

工具调用数: 144
工具返回数: 144
没有返回结果的调用数: 0
没有对应调用的返回数: 0


## 4. 构建SFT监督训练样本

In [14]:
from copy import deepcopy

# 合并连续的 Assistant 消息,保留文本内容和工具调用。
def merge_consecutive_assistants(messages):
    merged = []

    for message in messages:
        message = deepcopy(message)

        if (
            merged
            and message["role"] == "assistant"
            and merged[-1]["role"] == "assistant"
        ):
            prev = merged[-1]

            # 合并文本内容
            prev["content"] = (
                (prev.get("content") or "")
                + (message.get("content") or "")
            )

            # 合并工具调用
            if message.get("tool_calls"):
                prev.setdefault("tool_calls", []).extend(
                    message["tool_calls"]
                )

        else:
            merged.append(message)

    return merged

# 构建SFT训练样本，暂时不做上下文裁剪，也不使用Chat Template，仅保留结构化消息
def build_sft_examples(messages, source_file):
    examples = []
    history = []
    has_user = False

    # 先规范化消息
    messages = merge_consecutive_assistants(messages)

    for index, message in enumerate(messages):

        # 将完整的 Assistant 消息作为训练目标
        if message["role"] == "assistant" and has_user:
            examples.append({
                "source_file": source_file,
                "target_index": index,
                "prompt_messages": history.copy(),
                "target_message": message
            })

        # 当前消息进入历史
        history.append(message)

        if message["role"] == "user":
            has_user = True

    return examples


examples = build_sft_examples(
    messages,
    source_file=sample_file.name
)

print("生成的 SFT 样本数量:", len(examples))

生成的 SFT 样本数量: 156


In [15]:
# 打印前两条训练样本
for i, example in enumerate(examples[:2], start=1):
    print(f"\n========== 样本 {i} ==========")

    prompt = example["prompt_messages"]
    target = example["target_message"]

    print("Prompt 消息数量:", len(prompt))
    print("Prompt 角色顺序:", [m["role"] for m in prompt])

    if target.get("tool_calls"):
        print("Completion 类型: 工具调用")
        print(
            "工具名称:",
            [call["function"]["name"] for call in target["tool_calls"]]
        )
    else:
        print("Completion 类型: 文本回复")
        print("回复内容:", target["content"][:200])


========== 样本 1 ==========
Prompt 消息数量: 1
Prompt 角色顺序: ['user']
Completion 类型: 工具调用
工具名称: ['bash']

========== 样本 2 ==========
Prompt 消息数量: 3
Prompt 角色顺序: ['user', 'assistant', 'tool']
Completion 类型: 工具调用
工具名称: ['bash']


In [16]:
import json
import statistics

# 统计每条训练样本的历史消息数量
message_counts = [
    len(example["prompt_messages"])
    for example in examples
]

# 粗略统计 Prompt 的字符长度
prompt_lengths = [
    len(json.dumps(
        example["prompt_messages"],
        ensure_ascii=False
    ))
    for example in examples
]

print("训练样本数:", len(examples))
print("平均历史消息数:", round(statistics.mean(message_counts), 2))
print("最大历史消息数:", max(message_counts))
print("平均 Prompt 字符数:", round(statistics.mean(prompt_lengths)))
print("最大 Prompt 字符数:", max(prompt_lengths))

训练样本数: 156
平均历史消息数: 160.08
最大历史消息数: 318
平均 Prompt 字符数: 296634
最大 Prompt 字符数: 456296


In [17]:
longest_idx = prompt_lengths.index(max(prompt_lengths))
longest_example = examples[longest_idx]

print("样本索引:", longest_idx)
print("历史消息数量:", len(longest_example["prompt_messages"]))
print("目标消息类型:",
      "工具调用" if longest_example["target_message"].get("tool_calls")
      else "文本回复")

样本索引: 155
历史消息数量: 318
目标消息类型: 文本回复


In [18]:
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

model_id = "google/gemma-4-E2B-it"

# 仅下载 Tokenizer 相关文件
# tokenizer_dir = snapshot_download(
#     repo_id=model_id,
#     local_dir="./sft-data/gemma4-tokenizer",
#     allow_patterns=[
#         "tokenizer.json",
#         "tokenizer_config.json",
#         "chat_template.jinja",
#         "config.json"
#     ],
#     max_workers=1
# )

# print("Tokenizer 文件下载完成:", tokenizer_dir)

# 从本地文件加载，避免重复联网
tokenizer_dir="./sft-data/gemma4-tokenizer"
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_dir,
    local_files_only=True
)

print("Tokenizer 加载成功")
print("测试编码:", tokenizer.encode("Hello, world!"))

Tokenizer 加载成功
测试编码: [9259, 236764, 1902, 236888]


In [19]:
# 根据token数量截断文本
def clip_text_tokens(text, tokenizer, max_tokens=300):
    ids = tokenizer.encode(text, add_special_tokens=False)

    if len(ids) <= max_tokens:
        return text

    # 保留文本开头和结尾
    keep = (max_tokens - 20) // 2

    head = tokenizer.decode(ids[:keep], skip_special_tokens=False)
    tail = tokenizer.decode(ids[-keep:], skip_special_tokens=False)

    return head + "\n[...中间内容已省略...]\n" + tail

In [20]:
from collections import Counter

# 将工具调用和工具结果分组
def group_history(messages):
    groups = []
    i = 0

    while i < len(messages):
        msg = messages[i]

        if msg["role"] == "tool":
            raise ValueError("发现没有对应 Assistant 的工具结果")

        group = [msg]
        i += 1

        if msg["role"] == "assistant" and msg.get("tool_calls"):
            # 收集紧随其后的工具结果
            while i < len(messages) and messages[i]["role"] == "tool":
                group.append(messages[i])
                i += 1

            call_ids = [
                call["id"] for call in msg["tool_calls"]
            ]
            result_ids = [
                result["tool_call_id"] for result in group[1:]
            ]

            if Counter(call_ids) != Counter(result_ids):
                raise ValueError("工具调用与返回结果未完整匹配")

        groups.append(group)

    return groups

In [21]:
import json
from copy import deepcopy

# 使用JSON序列化后的token数量作为粗略估计
def estimate_tokens(messages, tokenizer):
    text = json.dumps(messages, ensure_ascii=False)
    return len(
        tokenizer.encode(text, add_special_tokens=False)
    )


def compact_message(message, tokenizer, max_tokens):
    msg = deepcopy(message)

    if msg["role"] == "tool":
        msg["content"] = clip_text_tokens(
            msg["content"], tokenizer, max_tokens
        )

    elif msg["role"] == "assistant":
        msg["content"] = clip_text_tokens(
            msg["content"], tokenizer, max_tokens
        )

    return msg


def trim_sft_context(example, tokenizer, token_budget=1200, max_tokens=300):
    history = example["prompt_messages"]

    # 找到最近一次 User 消息
    last_user_idx = max(
        i for i, msg in enumerate(history)
        if msg["role"] == "user"
    )

    # 保留当前任务的 User 消息
    user_message = deepcopy(history[last_user_idx])

    if estimate_tokens([user_message], tokenizer) > token_budget:
        return None

    # 对后续历史消息进行分组
    groups = group_history(history[last_user_idx + 1:])

    selected = []

    # 从最近的历史消息向前添加
    for group in reversed(groups):
        compact_group = [
            compact_message(msg, tokenizer, max_tokens)
            for msg in group
        ]

        candidate = [user_message] + compact_group + selected

        if estimate_tokens(candidate, tokenizer) > token_budget:
            break

        selected = compact_group + selected

    return [user_message] + selected

In [22]:
# 检查裁剪效果
example = examples[-1]

trimmed_context = trim_sft_context(
    example,
    tokenizer,
    token_budget=1200,
    max_tokens=300
)

if trimmed_context is None:
    print("最近的 User 消息本身已经超过预算")
else:
    print("原始消息数:", len(example["prompt_messages"]))
    print("裁剪后消息数:", len(trimmed_context))

    print(
        "裁剪后估算 Token 数:",
        estimate_tokens(trimmed_context, tokenizer)
    )

    print(
        "裁剪后角色分布:",
        Counter(msg["role"] for msg in trimmed_context)
    )

    # 检查工具调用和结果是否仍完整对应
    calls = [
        call["id"]
        for msg in trimmed_context
        for call in msg.get("tool_calls", [])
    ]

    results = [
        msg["tool_call_id"]
        for msg in trimmed_context
        if msg["role"] == "tool"
    ]

    print("工具调用和返回结果完整匹配:",
          Counter(calls) == Counter(results))
print(trimmed_context)

原始消息数: 318
裁剪后消息数: 3
裁剪后估算 Token 数: 452
裁剪后角色分布: Counter({'user': 1, 'assistant': 1, 'tool': 1})
工具调用和返回结果完整匹配: True
[{'role': 'user', 'content': 'yes, remove'}, {'role': 'assistant', 'content': '', 'tool_calls': [{'id': 'toolu_01BpkqXFysNB5wv2xmbwdFLq', 'type': 'function', 'function': {'name': 'bash', 'arguments': {'command': 'cd /Users/badlogic/workspaces/pi-mono && rm packages/coding-agent/examples/extensions/load-file.ts && git add -A && git commit -m "chore: remove test extension load-file.ts" && git push'}}}]}, {'role': 'tool', 'tool_call_id': 'toolu_01BpkqXFysNB5wv2xmbwdFLq', 'name': 'bash', 'content': 'Running formatting, linting, and type checking...\n\n> pi-monorepo@0.0.3 check\n> biome check --write . && tsgo --noEmit && npm run check -w @mariozechner/pi-web-ui\n\nChecked 405 files in 237ms. No fixes applied.\n\n> @mariozechner/pi-web-ui@0.46.0 check\n> biome check --write . && tsc --noEmit && cd example && biome check --write . && tsc --noEmit\n\nChecked 73 files in 37ms. N

In [23]:
history = longest_example["prompt_messages"]

last_user_idx = max(
    i for i, msg in enumerate(history)
    if msg["role"] == "user"
)

recent_history = history[last_user_idx + 1:]
groups = group_history(recent_history)

print("总历史消息数:", len(history))
print("最近一次 User 的位置:", last_user_idx)
print("该 User 之后的消息数:", len(recent_history))
print("后续完整分组数:", len(groups))

# 对比不同预算
for budget in [1200, 6000, 12000]:
    context = trim_sft_context(
        longest_example,
        tokenizer,
        token_budget=budget,
        max_tokens=300
    )

    if context is None:
        print(f"预算 {budget}: 无法保留 User 消息")
    else:
        print(
            f"预算 {budget}: "
            f"{len(context)} 条消息, "
            f"{estimate_tokens(context, tokenizer)} Tokens"
        )

总历史消息数: 318
最近一次 User 的位置: 315
该 User 之后的消息数: 2
后续完整分组数: 1
预算 1200: 3 条消息, 452 Tokens
预算 6000: 3 条消息, 452 Tokens
预算 12000: 3 条消息, 452 Tokens


In [24]:
history = longest_example["prompt_messages"]

user_positions = [
    i for i, msg in enumerate(history)
    if msg["role"] == "user"
]

# 检查最近几条User消息
for idx in user_positions[-5:]:
    print(f"\nUser 位置: {idx}")
    print("内容:", history[idx]["content"][:500])


User 位置: 286
内容: any other such fuck ups?

User 位置: 299
内容: page up/down now work

User 位置: 301
内容: yes, merge to main, close issue with comment in my tone

User 位置: 311
内容: i assume you removed the load-file.ts extension?

User 位置: 315
内容: yes, remove


In [25]:
# 按User消息划分轮次
def split_into_user_turns(history):
    turns = []

    for msg in history:
        if msg["role"] == "user":
            turns.append([msg])
        elif turns:
            turns[-1].append(msg)

    return turns


turns = split_into_user_turns(
    longest_example["prompt_messages"]
)

print("对话轮次数:", len(turns))
print("各轮消息数量:", [len(turn) for turn in turns])

对话轮次数: 19
各轮消息数量: [32, 5, 4, 2, 1, 134, 6, 10, 6, 2, 2, 32, 13, 37, 13, 2, 10, 4, 3]


In [26]:
from copy import deepcopy

# 根据轮次裁剪上下文
def trim_context_by_turns(example, tokenizer, token_budget=1200, max_tokens=300):
    history = example["prompt_messages"]
    turns = split_into_user_turns(history)

    if not turns:
        return None

    selected = []

    # 从最近的对话轮次开始保留
    for turn in reversed(turns):

        # 验证工具调用和结果的完整性
        groups = group_history(turn[1:])

        compact_turn = [deepcopy(turn[0])]

        for group in groups:
            compact_turn.extend(
                compact_message(msg, tokenizer, max_tokens)
                for msg in group
            )

        candidate = compact_turn + selected

        if estimate_tokens(candidate, tokenizer) > token_budget:
            # 最新轮次过长时，使用之前的组级裁剪
            if not selected:
                return trim_sft_context(
                    example, tokenizer, token_budget
                )

            # 不截断更早的轮次，保持历史连续
            break

        selected = candidate

    return selected

In [27]:
# 验证按照轮次裁剪
for budget in [1200, 6000, 12000]:
    context = trim_context_by_turns(
        longest_example,
        tokenizer,
        token_budget=budget,
        max_tokens=300
    )

    print(f"\n预算: {budget}")
    print("保留消息数:", len(context) if context else 0)

    if context:
        print(
            "估算 Token 数:",
            estimate_tokens(context, tokenizer)
        )
        print(
            "User 消息数:",
            sum(msg["role"] == "user" for msg in context)
        )


预算: 1200
保留消息数: 7
估算 Token 数: 682
User 消息数: 2

预算: 6000
保留消息数: 32
估算 Token 数: 4560
User 消息数: 5

预算: 12000
保留消息数: 69
估算 Token 数: 11453
User 消息数: 6


## 5. Chat Template与Prompt/Completion渲染

In [28]:
# 检查 Tokenizer 是否包含 Chat Template
assert tokenizer.chat_template is not None, "当前 Tokenizer 没有加载 Chat Template"

test_messages = [
    {"role": "user", "content": "hello"},
    {"role": "assistant", "content": "Hello!"}
]

# 测试最简单的Chat Template
text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False
)

print(text)

<bos><|turn>user
hello<turn|>
<|turn>model
Hello!<turn|>



In [29]:
# 为了减少复杂度，先选择一条最简单的bash工具调用样本
tool_example = next(
    ex for ex in examples
    if len(ex["prompt_messages"]) == 1
    and ex["target_message"].get("tool_calls")
    and ex["target_message"]["tool_calls"][0]["function"]["name"] == "bash"
)

print("Prompt:", tool_example["prompt_messages"])
print("Target:", tool_example["target_message"])

# 定义bash工具
bash_tool = [{
    "type": "function",
    "function": {
        "name": "bash",
        "description": "Run a shell command in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "Shell command to execute."
                }
            },
            "required": ["command"]
        }
    }
}]

# 渲染模型的输入上下文
prompt_text = tokenizer.apply_chat_template(
    tool_example["prompt_messages"],
    tools=bash_tool,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# 渲染包含目标 Assistant 消息的完整对话
full_text = tokenizer.apply_chat_template(
    tool_example["prompt_messages"] + [
        tool_example["target_message"]
    ],
    tools=bash_tool,
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False
)

# 确认 Prompt 是完整对话的前缀
assert full_text.startswith(prompt_text), "Chat Template 前缀不一致"

# 提取监督目标
completion_text = full_text[len(prompt_text):]

print("========== PROMPT ==========")
print(prompt_text)

print("\n========== COMPLETION ==========")
print(completion_text)

Prompt: [{'role': 'user', 'content': 'Analyze GitHub issue(s): https://github.com/badlogic/pi-mono/issues/732 you will have to pull downt he image and read it as well to understand.\n\nFor each issue:\n\n1. Read the issue in full, including all comments and linked issues/PRs.\n\n2. **For bugs**:\n   - Ignore any root cause analysis in the issue (likely wrong)\n   - Read all related code files in full (no truncation)\n   - Trace the code path and identify the actual root cause\n   - Propose a fix\n\n3. **For feature requests**:\n   - Read all related code files in full (no truncation)\n   - Propose the most concise implementation approach\n   - List affected files and changes needed\n\nDo NOT implement unless explicitly asked. Analyze and propose only.'}]
Target: {'role': 'assistant', 'content': '', 'tool_calls': [{'id': 'toolu_01HK4MwpXYrA79PdGFxamC1S', 'type': 'function', 'function': {'name': 'bash', 'arguments': {'command': 'gh issue view 732 --repo badlogic/pi-mono --json title,body

In [30]:
# 检查token实际长度
prompt_ids = tokenizer.encode(
    prompt_text,
    add_special_tokens=False
)

full_ids = tokenizer.encode(
    full_text,
    add_special_tokens=False
)

print("Prompt Token 数:", len(prompt_ids))
print("完整序列 Token 数:", len(full_ids))

# 检查 Token 级别的前缀是否一致
prefix_match = full_ids[:len(prompt_ids)] == prompt_ids
print("Token 前缀一致:", prefix_match)

if prefix_match:
    print("Completion Token 数:", len(full_ids) - len(prompt_ids))
else:
    print("分词边界不一致，后续需要重新处理 Loss Mask 的对齐")

print("是否超过2048:", len(full_ids) > 2048)

Prompt Token 数: 240
完整序列 Token 数: 278
Token 前缀一致: True
Completion Token 数: 38
是否超过2048: False


In [31]:
# 统计当前session实际使用过哪些工具
from collections import Counter

tool_counts = Counter(
    call["function"]["name"]
    for msg in messages
    for call in msg.get("tool_calls", [])
)

print("当前 Session 的工具调用统计：")
print(tool_counts)

当前 Session 的工具调用统计：
Counter({'bash': 66, 'read': 47, 'edit': 30, 'write': 1})


In [32]:
from typing import Any

# Agent工具定义
KNOWN_TOOL_SCHEMAS: dict[str, dict[str, Any]] = {
    "bash": {
        "description": "Run a shell command in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "Shell command to run."},
                "cmd": {"type": "string", "description": "Shell command to run."},
                "timeout": {"type": "number", "description": "Optional timeout in milliseconds."},
            },
            "required": [],
        },
    },
    "read": {
        "description": "Read a file or image from the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path to read."},
                "file": {"type": "string", "description": "Path to read."},
                "offset": {"type": "number", "description": "Optional starting line."},
                "limit": {"type": "number", "description": "Optional line limit."},
                "start": {"type": "number", "description": "Optional starting line."},
                "end": {"type": "number", "description": "Optional ending line."},
            },
            "required": [],
        },
    },
    "edit": {
        "description": "Edit a file in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path to edit."},
                "oldText": {"type": "string", "description": "Text to replace."},
                "newText": {"type": "string", "description": "Replacement text."},
                "edits": {"type": "array", "description": "Structured edits."},
                "patch": {"type": "string", "description": "Patch content."},
            },
            "required": [],
        },
    },
    "write": {
        "description": "Write content to a file in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path to write."},
                "content": {"type": "string", "description": "File content."},
            },
            "required": [],
        },
    },
    "grep": {
        "description": "Search text in files.",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Search pattern."},
                "path": {"type": "string", "description": "Path to search."},
                "limit": {"type": "number", "description": "Optional result limit."},
                "literal": {"type": "boolean", "description": "Treat pattern literally."},
                "context": {"type": "number", "description": "Context lines."},
            },
            "required": [],
        },
    },
    "find": {
        "description": "Find files or text in the workspace.",
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "Pattern to find."},
                "path": {"type": "string", "description": "Path to search."},
                "limit": {"type": "number", "description": "Optional result limit."},
            },
            "required": [],
        },
    },
    "ls": {
        "description": "List files in a directory.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Directory path."},
                "limit": {"type": "number", "description": "Optional result limit."},
            },
            "required": [],
        },
    },
    "todo": {
        "description": "Manage a lightweight task list.",
        "parameters": {
            "type": "object",
            "properties": {
                "action": {"type": "string", "description": "Task-list action."},
                "text": {"type": "string", "description": "Task text."},
                "id": {"type": "string", "description": "Task identifier."},
            },
            "required": [],
        },
    },
}

In [33]:
# 生成默认工具定义
def generic_tool_schema(name: str) -> dict[str, Any]:
    return {
        "description": f"Pi coding-agent tool named {name}.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    }

# 生成工具列表
def make_tool_definitions(tool_names: set[str]) -> list[dict[str, Any]]:
    tools: list[dict[str, Any]] = []
    for name in sorted(n for n in tool_names if n):
        schema = KNOWN_TOOL_SCHEMAS.get(name, generic_tool_schema(name))
        tools.append({"type": "function", "function": {"name": name, **schema}})
    return tools

In [34]:
# 选择最后一条训练样本
example = examples[-1]

# 使用之前的轮次裁剪函数
context = trim_context_by_turns(
    example,
    tokenizer,
    token_budget=1600,
    max_tokens=300
)

# 构建当前 Session 的工具定义
tools = make_tool_definitions(set(tool_counts))

# 渲染 Prompt
prompt = tokenizer.apply_chat_template(
    context,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

# 渲染完整序列
full = tokenizer.apply_chat_template(
    context + [example["target_message"]],
    tools=tools,
    tokenize=False,
    add_generation_prompt=False,
    enable_thinking=False
)

# 检查文本前缀
assert full.startswith(prompt), "Prompt 前缀不一致"

completion = full[len(prompt):]

# Tokenization
prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
full_ids = tokenizer.encode(full, add_special_tokens=False)

prefix_match = full_ids[:len(prompt_ids)] == prompt_ids

print("Prompt 历史消息数:", len(context))
print("Prompt Token 数:", len(prompt_ids))
print("完整序列 Token 数:", len(full_ids))
print("Token 前缀一致:", prefix_match)
print("是否超过2048:", len(full_ids) > 2048)

if prefix_match:
    print("Completion Token 数:", len(full_ids) - len(prompt_ids))

print("\nCompletion:")
print(completion)

Prompt 历史消息数: 7
Prompt Token 数: 874
完整序列 Token 数: 878
Token 前缀一致: True
是否超过2048: False
Completion Token 数: 4

Completion:
Done.<turn|>



In [35]:
from collections import Counter

# 验证每轮的工具调用是否完整
def validate_tool_pairs(messages):
    pending = Counter()

    for msg in messages:
        if msg["role"] == "assistant":
            for call in msg.get("tool_calls", []):
                pending[call["id"]] += 1

        elif msg["role"] == "tool":
            call_id = msg.get("tool_call_id")

            if not call_id or pending[call_id] <= 0:
                return False

            pending[call_id] -= 1

    return all(count == 0 for count in pending.values())

In [36]:
# 渲染消息
def render_messages(messages, tokenizer, tools):
    return tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

# 渲染完整对话
def render_full_conversation(messages, tokenizer, tools):
    return tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

# 计算token数
def count_tokens(text, tokenizer):
    return len(
        tokenizer.encode(
            text,
            add_special_tokens=False
        )
    )

In [37]:
# 实现动态token预算裁剪器
from copy import deepcopy

# 修复prompt_text 在某处输出了 <turn|>，而 full_text 在同一位置继续输出工具调用。
def render_aligned_pair(messages, target, tokenizer, tools):
    prompt_text = render_messages(
        messages, tokenizer, tools
    )

    full_text = render_full_conversation(
        messages + [target], tokenizer, tools
    )

    # 正常情况：Prompt 本来就是完整序列的前缀
    if full_text.startswith(prompt_text):
        pass

    # Gemma 4 可能提前结束了 Assistant 轮次
    elif (
        prompt_text.endswith("<turn|>\n")
        and full_text.startswith(
            prompt_text[:-len("<turn|>\n")]
        )
    ):
        # 仅移除这个多余的轮次结束标记
        prompt_text = prompt_text[:-len("<turn|>\n")]

    else:
        # 其他不一致情况仍然报错，不静默修改
        i = 0
        for a, b in zip(prompt_text, full_text):
            if a != b:
                break
            i += 1

        print("第一个不一致位置:", i)
        print("Prompt:", repr(prompt_text[max(0, i-80):i+100]))
        print("Full:  ", repr(full_text[max(0, i-80):i+100]))

        raise ValueError("无法对齐 Prompt 和 Full")

    # Token 级前缀验证
    prompt_ids = tokenizer.encode(
        prompt_text, add_special_tokens=False
    )

    full_ids = tokenizer.encode(
        full_text, add_special_tokens=False
    )

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("Token 前缀不一致")

    return prompt_text, full_text, prompt_ids, full_ids

def build_rendered_sft_example(
    example,
    tokenizer,
    tools,
    max_length=2048
):
    history = example["prompt_messages"]
    target = deepcopy(example["target_message"])

    if not any(msg["role"] == "user" for msg in history):
        return None

    # 目标消息必须是 Assistant
    if target["role"] != "assistant":
        return None

    turns = split_into_user_turns(history)

    # 从最近的轮次向前尝试添加上下文
    selected = []

    for turn in reversed(turns):
        candidate = turn + selected

        prompt_text, full_text, prompt_ids, full_ids = render_aligned_pair(
            candidate,
            target,
            tokenizer,
            tools
        )

        if len(full_ids) <= max_length:
            selected = candidate
        else:
            # 不拆分对话轮次
            break

    if not selected:
        return None

    prompt_text, full_text, prompt_ids, full_ids = render_aligned_pair(
        selected,
        target,
        tokenizer,
        tools
    )

    if len(full_ids) > max_length:
        return None

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("最终 Token 前缀校验失败")

    return {
        "prompt": prompt_text,
        "completion": full_text[len(prompt_text):],
        "prompt_tokens": len(prompt_ids),
        "completion_tokens": len(full_ids) - len(prompt_ids),
        "total_tokens": len(full_ids),
        "context_messages": selected,
        "target_message": target
    }

In [38]:
# 测试上下文裁剪
tools = make_tool_definitions(
    set(KNOWN_TOOL_SCHEMAS.keys())
)

# 先测试最长历史样本
test_example = examples[-1]

result = build_rendered_sft_example(
    test_example,
    tokenizer,
    tools,
    max_length=2048
)

if result is None:
    print("该样本无法在2048 Token限制内保留有效上下文")
else:
    print("Prompt Tokens:", result["prompt_tokens"])
    print("Completion Tokens:", result["completion_tokens"])
    print("Total Tokens:", result["total_tokens"])
    print("保留消息数:", len(result["context_messages"]))
    print("是否超过限制:", result["total_tokens"] > 2048)

    print("\nPrompt 预览:")
    print(result["prompt"][:1000])

    print("\nCompletion 预览:")
    print(result["completion"][:500])

Prompt Tokens: 1201
Completion Tokens: 4
Total Tokens: 1205
保留消息数: 7
是否超过限制: False

Prompt 预览:
<bos><|turn>system
<|tool>declaration:bash{description:<|"|>Run a shell command in the workspace.<|"|>,parameters:{properties:{cmd:{description:<|"|>Shell command to run.<|"|>,type:<|"|>STRING<|"|>},command:{description:<|"|>Shell command to run.<|"|>,type:<|"|>STRING<|"|>},timeout:{description:<|"|>Optional timeout in milliseconds.<|"|>,type:<|"|>NUMBER<|"|>}},type:<|"|>OBJECT<|"|>}}<tool|><|tool>declaration:edit{description:<|"|>Edit a file in the workspace.<|"|>,parameters:{properties:{edits:{description:<|"|>Structured edits.<|"|>,type:<|"|>ARRAY<|"|>},newText:{description:<|"|>Replacement text.<|"|>,type:<|"|>STRING<|"|>},oldText:{description:<|"|>Text to replace.<|"|>,type:<|"|>STRING<|"|>},patch:{description:<|"|>Patch content.<|"|>,type:<|"|>STRING<|"|>},path:{description:<|"|>Path to edit.<|"|>,type:<|"|>STRING<|"|>}},type:<|"|>OBJECT<|"|>}}<tool|><|tool>declaration:find{description:

In [39]:
# 构建最终数据集
rendered_examples = []
dropped_examples = 0

for example in examples:
    result = build_rendered_sft_example(
        example,
        tokenizer,
        tools,
        max_length=2048
    )

    if result is None:
        dropped_examples += 1
        continue

    rendered_examples.append({
        "prompt": result["prompt"],
        "completion": result["completion"]
    })

print("原始样本数:", len(examples))
print("成功保留:", len(rendered_examples))
print("过滤样本数:", dropped_examples)
lengths = [
    len(tokenizer.encode(
        item["prompt"] + item["completion"],
        add_special_tokens=False
    ))
    for item in rendered_examples
]

print("最终样本数:", len(lengths))

if lengths:
    print("最短序列:", min(lengths))
    print("最长序列:", max(lengths))
    print("平均序列:", round(sum(lengths) / len(lengths), 2))

原始样本数: 156
成功保留: 43
过滤样本数: 113
最终样本数: 43
最短序列: 772
最长序列: 2011
平均序列: 1380.65


In [40]:
# 实验max_length应该如何设置
max_lengths = [2048, 4096, 8192, 16384]

results = []

for max_length in max_lengths:
    kept = 0
    dropped = 0
    token_lengths = []

    for example in examples:
        result = build_rendered_sft_example(
            example,
            tokenizer,
            tools,
            max_length=max_length
        )

        if result is None:
            dropped += 1
        else:
            kept += 1
            token_lengths.append(result["total_tokens"])

    results.append({
        "max_length": max_length,
        "kept": kept,
        "dropped": dropped,
        "retention_rate": round(kept / len(examples) * 100, 2),
        "avg_tokens": (
            round(sum(token_lengths) / len(token_lengths), 2)
            if token_lengths else 0
        )
    })

import pandas as pd

display(pd.DataFrame(results))

,max_length,kept,dropped,retention_rate,avg_tokens
0,2048,43,113,27.56,1380.65
1,4096,60,96,38.46,2318.20
2,8192,78,78,50.00,4770.67
3,16384,101,55,64.74,7567.72


### 上下文裁剪问题与后续优化方向

在处理 pi-mono Agent 轨迹数据时，部分 User 轮次包含大量工具调用及其返回结果，导致历史上下文达到数万 Token。当前采用按完整 User 轮次裁剪的策略，不允许在轮次内部截断，因此当最新轮次超过最大序列长度时，整个 SFT 样本会被丢弃。

实验表明，当 `max_length=8192` 时，156 条训练样本中保留了 78 条，保留率为 50%。后续可以通过按完整工具调用与返回结果进行分组裁剪、保留最近任务上下文等方式，提高样本利用率，同时确保工具调用结构完整。

现阶段暂时采用 **8192 Token** 的最大序列长度，优先完成 SFT 数据准备、模型加载、LoRA 微调、损失计算及训练评估的完整流程。

## 6. 构建STF训练数据集

In [41]:
# 将渲染后的SFT样本转换为token级训练数据
def tokenize_sft_example(result, tokenizer):
    prompt_text = result["prompt"]
    completion_text = result["completion"]

    # 拼接完整训练序列
    full_text = prompt_text + completion_text

    # 对完整序列 Tokenize
    full_ids = tokenizer.encode(
        full_text,
        add_special_tokens=False
    )

    # 对 Prompt Tokenize
    prompt_ids = tokenizer.encode(
        prompt_text,
        add_special_tokens=False
    )

    # 检查 Token 前缀一致性
    assert full_ids[:len(prompt_ids)] == prompt_ids

    # 只有 Completion 部分参与 Loss
    labels = (
        [-100] * len(prompt_ids)
        + full_ids[len(prompt_ids):]
    )

    # 当前没有padding，全部token有效
    attention_mask = [1] * len(full_ids)

    return {
        "input_ids": full_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [42]:
MAX_LENGTH = 8192

# 使用第一条能够成功渲染的样本
result = next(
    (
        r for example in examples
        if (r := build_rendered_sft_example(
            example,
            tokenizer,
            tools,
            max_length=MAX_LENGTH
        )) is not None
    ),
    None
)

assert result is not None, "没有可用的训练样本"

sample = tokenize_sft_example(result, tokenizer)

print("input_ids:", len(sample["input_ids"]))
print("attention_mask:", len(sample["attention_mask"]))
print("labels:", len(sample["labels"]))

print(
    "参与 Loss 的 Token 数:",
    sum(x != -100 for x in sample["labels"])
)

print(
    "被 Mask 的 Token 数:",
    sum(x == -100 for x in sample["labels"])
)

input_ids: 943
attention_mask: 943
labels: 943
参与 Loss 的 Token 数: 38
被 Mask 的 Token 数: 905


In [43]:
# 构建huggingface dataset
from datasets import Dataset

MAX_LENGTH = 2048

tokenized_examples = []
dropped_count = 0

for example in examples:

    # 第 5 节：动态裁剪和渲染
    result = build_rendered_sft_example(
        example,
        tokenizer,
        tools,
        max_length=MAX_LENGTH
    )

    if result is None:
        dropped_count += 1
        continue

    # 第 6 节：构建 input_ids / attention_mask / labels
    tokenized = tokenize_sft_example(
        result,
        tokenizer
    )

    tokenized_examples.append(tokenized)

# 转换为 Hugging Face Dataset
train_dataset = Dataset.from_list(tokenized_examples)

print("原始样本数:", len(examples))
print("保留样本数:", len(train_dataset))
print("过滤样本数:", dropped_count)

print("\nDataset 结构:")
print(train_dataset)

原始样本数: 156
保留样本数: 43
过滤样本数: 113

Dataset 结构:
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 43
})


In [44]:
# 自定义Data Collator，将同一批次的数据填充到同一长度
import torch

class SFTDataCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

        # 如果 Tokenizer 没有定义 PAD，则使用 EOS 作为填充值
        if self.pad_token_id is None:
            self.pad_token_id = tokenizer.eos_token_id

        if self.pad_token_id is None:
            raise ValueError("Tokenizer 没有可用的 PAD/EOS Token")

    def __call__(self, features):
        # 当前 Batch 中最长序列
        max_length = max(
            len(feature["input_ids"])
            for feature in features
        )

        batch_input_ids = []
        batch_attention_mask = []
        batch_labels = []

        for feature in features:
            input_ids = feature["input_ids"]
            attention_mask = feature["attention_mask"]
            labels = feature["labels"]

            assert len(input_ids) == len(attention_mask) == len(labels)

            pad_length = max_length - len(input_ids)

            # 右侧 Padding
            batch_input_ids.append(
                input_ids
                + [self.pad_token_id] * pad_length
            )

            batch_attention_mask.append(
                attention_mask
                + [0] * pad_length
            )

            batch_labels.append(
                labels
                + [-100] * pad_length
            )

        return {
            "input_ids": torch.tensor(
                batch_input_ids, dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                batch_attention_mask, dtype=torch.long
            ),
            "labels": torch.tensor(
                batch_labels, dtype=torch.long
            )
        }

data_collator = SFTDataCollator(tokenizer)

In [45]:
# 验证动态 Padding

# 找出最短和最长的样本，用来测试 Padding
lengths = [
    len(example["input_ids"])
    for example in train_dataset
]

short_idx = min(range(len(lengths)), key=lambda i: lengths[i])
long_idx = max(range(len(lengths)), key=lambda i: lengths[i])

features = [
    train_dataset[short_idx],
    train_dataset[long_idx]
]

batch = data_collator(features)

print("原始长度:")
print([len(x["input_ids"]) for x in features])

print("\nBatch Shape:")
for key, value in batch.items():
    print(f"{key}: {value.shape}")

print("\n每条样本的有效 Token 数:")
print(batch["attention_mask"].sum(dim=1))

print("\n每条样本参与 Loss 的 Token 数:")
print((batch["labels"] != -100).sum(dim=1))

原始长度:
[772, 2011]

Batch Shape:
input_ids: torch.Size([2, 2011])
attention_mask: torch.Size([2, 2011])
labels: torch.Size([2, 2011])

每条样本的有效 Token 数:
tensor([ 772, 2011])

每条样本参与 Loss 的 Token 数:
tensor([24, 57])


In [46]:
# 三个字段 Shape 完全一致
assert (
    batch["input_ids"].shape
    == batch["attention_mask"].shape
    == batch["labels"].shape
)

# 所有 Padding 位置必须被 Mask
padding_positions = batch["attention_mask"] == 0

assert torch.all(
    batch["labels"][padding_positions] == -100
)

# 每条样本的有效 Token 必须保持原样
for i, feature in enumerate(features):
    n = len(feature["input_ids"])

    assert batch["input_ids"][i, :n].tolist() == feature["input_ids"]

    assert batch["labels"][i, :n].tolist() == feature["labels"]

print("Data Collator 验证通过！")

Data Collator 验证通过！


## 7. 加载预训练模型与配置LoRA

In [47]:
# 检查运行环境
import torch
import transformers
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

print("\nCUDA 可用:", torch.cuda.is_available())
print("GPU 数量:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {props.name}, "
        f"显存 {props.total_memory / 1024**3:.1f} GiB"
    )

print("BF16 支持:", torch.cuda.is_bf16_supported())

PyTorch: 2.10.0+cu128
Transformers: 5.5.0
PEFT: 0.21.2
Accelerate: 1.15.0

CUDA 可用: True
GPU 数量: 1
GPU 0: NVIDIA GeForce RTX 5090, 显存 31.4 GiB
BF16 支持: True


In [48]:
# 加载Gemma 4 E2B模型
from transformers import AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-E2B-it"

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
    low_cpu_mem_usage=True
)

print("模型类型:", type(model).__name__)
print("模型参数类型:", next(model.parameters()).dtype)
print("模型所在设备:", next(model.parameters()).device)

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

模型类型: Gemma4ForConditionalGeneration
模型参数类型: torch.bfloat16
模型所在设备: cuda:0


In [49]:
# 查看语言模型中的注意力投影层
for name, module in model.named_modules():
    if (
        "language_model.layers." in name
        and name.endswith((".q_proj", ".v_proj"))
    ):
        print(name, type(module).__name__)
        if name.endswith(".v_proj"):
            break

model.language_model.layers.0.self_attn.q_proj Linear
model.language_model.layers.0.self_attn.v_proj Linear


In [50]:
# 配置LoRA
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,

    # 暂时只训练文本 Attention 中的 Q/V 投影
    target_modules=(
        r".*\.language_model\.layers\.\d+"
        r"\.self_attn\.(q_proj|v_proj)$"
    )
)

model = get_peft_model(model, lora_config)

In [51]:
# 验证LoRA是否成功

# 打印可训练参数统计
model.print_trainable_parameters()

# 检查实际训练的参数
trainable_params = [
    (name, param)
    for name, param in model.named_parameters()
    if param.requires_grad
]

print("\n可训练参数张量数量:", len(trainable_params))

print("\n前 10 个可训练参数:")
for name, param in trainable_params[:10]:
    print(name, tuple(param.shape))

# 确保只有 LoRA 参数参与训练
assert len(trainable_params) > 0

assert all(
    "lora_" in name
    for name, _ in trainable_params
)

# 确保没有训练视觉和音频模块
assert all(
    "vision_tower" not in name
    and "audio_tower" not in name
    for name, _ in trainable_params
)

print("\nLoRA 配置验证通过！")

trainable params: 1,634,304 || all params: 5,124,812,320 || trainable%: 0.0319

可训练参数张量数量: 140

前 10 个可训练参数:
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight (8, 1536)
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight (2048, 8)
base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_A.default.weight (8, 1536)
base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_B.default.weight (256, 8)
base_model.model.model.language_model.layers.1.self_attn.q_proj.lora_A.default.weight (8, 1536)
base_model.model.model.language_model.layers.1.self_attn.q_proj.lora_B.default.weight (2048, 8)
base_model.model.model.language_model.layers.1.self_attn.v_proj.lora_A.default.weight (8, 1536)
base_model.model.model.language_model.layers.1.self_attn.v_proj.lora_B.default.weight (256, 8)
base_model.model.model.language_model.layers.2.self_attn.q_proj.lora_A.default.weight (8, 1536)
base_model.model.model.langua

In [52]:
# 检查加载模型后的显存
print(
    "GPU 0 当前显存分配:",
    round(torch.cuda.memory_allocated(0) / 1024**3, 2),
    "GiB"
)

print(
    "GPU 0 显存峰值:",
    round(torch.cuda.max_memory_allocated(0) / 1024**3, 2),
    "GiB"
)

GPU 0 当前显存分配: 9.55 GiB
GPU 0 显存峰值: 9.55 GiB


## 8. 配置SFT训练参数与Trainer

In [53]:
# 训练超参数
TRAIN_CONFIG = {
    "max_length": 8192,
    # 每次处理一条样本，但累计四次反向传播得到的梯度后，才更新一次参数
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "num_train_epochs": 3,
    "max_grad_norm": 1.0,
    "warmup_ratio": 0.03,
    "seed": 42,
}

In [54]:
# 启用Gradient Checkpointing以优化显存
model.config.use_cache = False
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    }
)
print("Gradient Checkpointing 已启用")

Gradient Checkpointing 已启用


In [55]:
# 准备一个测试 Batch
short_idx = min(
    range(len(train_dataset)),
    key=lambda i: len(train_dataset[i]["input_ids"])
)

batch = data_collator([
    train_dataset[short_idx]
])

device = next(model.parameters()).device

batch = {
    key: value.to(device)
    for key, value in batch.items()
}

print("输入 Shape:", batch["input_ids"].shape)
print("参与 Loss 的 Tokens:", (batch["labels"] != -100).sum().item())

输入 Shape: torch.Size([1, 772])
参与 Loss 的 Tokens: 24


In [56]:
# 创建AdamW优化器
optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=TRAIN_CONFIG["learning_rate"],
    weight_decay=TRAIN_CONFIG["weight_decay"]
)

In [57]:
# 一次完整的参数更新

model.train()
optimizer.zero_grad(set_to_none=True)

# 保存一个LoRA B矩阵，用来验证参数是否发生更新
check_name, check_param = next(
    (name, param)
    for name, param in model.named_parameters()
    if param.requires_grad and "lora_B" in name
)

before = check_param.detach().clone()

# Forward
outputs = model(**batch)
loss = outputs.loss

print("训练 Loss:", loss.item())

assert torch.isfinite(loss), "Loss 出现 NaN 或 Inf"

# Backward
loss.backward()

# 检查梯度
grad_count = sum(
    p.grad is not None
    for p in model.parameters()
    if p.requires_grad
)

print("具有梯度的参数张量数:", grad_count)

# 梯度裁剪
grad_norm = torch.nn.utils.clip_grad_norm_(
    (p for p in model.parameters() if p.requires_grad),
    max_norm=TRAIN_CONFIG["max_grad_norm"]
)

print("裁剪前梯度范数:", grad_norm.item())

# 更新 LoRA 参数
optimizer.step()

# 检查参数是否变化
changed = not torch.equal(before, check_param.detach())

print("检查参数:", check_name)
print("参数是否发生更新:", changed)

assert changed, "LoRA 参数没有发生更新"

optimizer.zero_grad(set_to_none=True)

print("单步 SFT 训练验证通过！")

训练 Loss: 12.475776672363281
具有梯度的参数张量数: 140
裁剪前梯度范数: 28.439605712890625
检查参数: base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight
参数是否发生更新: True
单步 SFT 训练验证通过！


In [58]:
print(
    "GPU 峰值显存:",
    round(torch.cuda.max_memory_allocated(0) / 1024**3, 2),
    "GiB"
)

GPU 峰值显存: 13.07 GiB


In [59]:
# 准备 Smoke Test Dataset

lengths = [
    len(item["input_ids"])
    for item in train_dataset
]

short_indices = sorted(
    range(len(lengths)),
    key=lambda i: lengths[i]
)[:8]

smoke_dataset = train_dataset.select(short_indices)

print("测试样本数:", len(smoke_dataset))
print(
    "序列长度:",
    [len(x["input_ids"]) for x in smoke_dataset]
)

测试样本数: 8
序列长度: [772, 781, 823, 827, 836, 870, 890, 943]


In [60]:
# 配置TrainingArguments
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./sft-output/gemma4-e2b-lora-smoke",

    # Batch Size
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # 训练轮次
    num_train_epochs=1,

    # 优化器
    learning_rate=2e-4,
    weight_decay=0.01,
    optim="adamw_torch",
    max_grad_norm=1.0,

    # 精度与显存优化
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    # 学习率调度
    lr_scheduler_type="cosine",
    warmup_ratio=0.0,

    # 日志
    logging_strategy="steps",
    logging_steps=1,
    report_to="none",

    # 暂不评估与保存
    eval_strategy="no",
    save_strategy="no",

    # 保留自定义 Dataset 字段
    remove_unused_columns=False,

    seed=42,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [61]:
# 创建 Trainer

model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=smoke_dataset,
    data_collator=data_collator,
)

# 使用模型返回的标准平均 Loss
trainer.model_accepts_loss_kwargs = False

print("Trainer 创建成功")
print("训练样本数:", len(trainer.train_dataset))

Trainer 创建成功
训练样本数: 8


In [62]:
# 训练冒烟测试

torch.cuda.reset_peak_memory_stats(0)

train_result = trainer.train()

print("\n训练结果:")
print(train_result.metrics)

print("\nOptimizer Step 数:", trainer.state.global_step)

print(
    "训练峰值显存:",
    round(torch.cuda.max_memory_allocated(0) / 1024**3, 2),
    "GiB"
)

Step,Training Loss
1,14.919230
2,14.236084



训练结果:
{'train_runtime': 3.9316, 'train_samples_per_second': 2.035, 'train_steps_per_second': 0.509, 'total_flos': 96006709540992.0, 'train_loss': 14.577657222747803, 'epoch': 1.0}

Optimizer Step 数: 2
训练峰值显存: 13.72 GiB


In [63]:
print("CUDA 可见 GPU 数:", torch.cuda.device_count())
print("Trainer n_gpu:", trainer.args.n_gpu)
print("Trainer train_batch_size:", trainer.args.train_batch_size)
print("Trainer parallel_mode:", trainer.args.parallel_mode)
print("Gradient accumulation:", trainer.args.gradient_accumulation_steps)
print("DataLoader Batch 数:", len(trainer.get_train_dataloader()))

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i} 峰值显存:",
        round(torch.cuda.max_memory_allocated(i) / 1024**3, 2),
        "GiB"
    )

CUDA 可见 GPU 数: 1
Trainer n_gpu: 1
Trainer train_batch_size: 1
Trainer parallel_mode: ParallelMode.NOT_PARALLEL
Gradient accumulation: 4
DataLoader Batch 数: 8
GPU 0 峰值显存: 13.72 GiB


## 9. 执行SFT训练并监控Loss

In [64]:
# 测试不同序列长度的训练显存

import gc
import torch


def test_training_memory(model, dataset, data_collator, max_tokens):
    # 选择不超过 max_tokens 的最长样本
    candidates = [
        i for i in range(len(dataset))
        if len(dataset[i]["input_ids"]) <= max_tokens
    ]

    if not candidates:
        print(f"没有长度 <= {max_tokens} 的样本")
        return None

    idx = max(
        candidates,
        key=lambda i: len(dataset[i]["input_ids"])
    )

    sample = dataset[idx]
    batch = data_collator([sample])

    batch = {
        k: v.to("cuda:0")
        for k, v in batch.items()
    }

    seq_len = batch["input_ids"].shape[1]

    model.train()
    model.zero_grad(set_to_none=True)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(0)

    outputs = None
    loss = None

    try:
        # Forward
        outputs = model(**batch)
        loss = outputs.loss

        # Backward
        loss.backward()
        torch.cuda.synchronize()

        peak_memory = (
            torch.cuda.max_memory_allocated(0) / 1024**3
        )

        result = {
            "seq_len": seq_len,
            "loss": loss.item(),
            "peak_memory_gib": round(peak_memory, 2)
        }

        print(result)
        return result

    finally:
        # 测试不执行 optimizer.step()
        # 因此不会修改模型权重
        model.zero_grad(set_to_none=True)

        del outputs, loss, batch
        gc.collect()
        torch.cuda.empty_cache()

In [65]:
# 测试max_length为2048
# result_2048 = test_training_memory(
#     model,
#     train_dataset,
#     data_collator,
#     max_tokens=2048
# )

In [66]:
# 测试max_length为4096
# result_4096 = test_training_memory(
#     model,
#     train_dataset,
#     data_collator,
#     max_tokens=4096
# )

In [67]:
# 测试max_length为8192
# result_8192 = test_training_memory(
#     model,
#     train_dataset,
#     data_collator,
#     max_tokens=8192
# )

### 不同 max_length 下的显存测试与结果分析

在单张 RTX 5090（32GB）上，使用 Gemma 4 E2B、BF16 精度、LoRA 微调和 Gradient Checkpointing，对不同序列长度进行 Forward 和 Backward 显存测试。结果表明：当 `max_length=2048` 时，实际测试序列为 1997 Tokens，峰值显存为 **19.45 GiB**，训练正常；当 `max_length=4096` 时，实际序列为 4045 Tokens，峰值显存达到 **28.79 GiB**，虽然训练成功，但显存余量较小；当 `max_length=8192` 时，训练发生 CUDA OOM，报错位置位于 Gemma 4 的 Loss 计算阶段，需要额外分配约 8 GiB 显存，但剩余显存不足。主要原因是长序列下完整 Logits 张量及其相关计算占用大量显存，即使 Prompt 部分通过 `labels=-100` 屏蔽 Loss，也不会自动避免这些 Logits 的计算。因此，现阶段暂时选择 **2048 Tokens** 完成 SFT 训练流程，后续可通过优化 Logits/Loss 计算、调整上下文长度或采用 QLoRA 等技术，进一步探索 8192 Tokens 的训练方案。

In [68]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./sft-output/gemma4-e2b-lora-sft",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    num_train_epochs=3,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,

    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    logging_steps=1,
    logging_strategy="steps",
    report_to="none",

    eval_strategy="no",
    save_strategy="no",

    remove_unused_columns=False,
    seed=42,
)

model.config.use_cache = False

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

trainer.model_accepts_loss_kwargs = False

train_result = trainer.train()

print("训练结果:", train_result.metrics)
print("Optimizer Steps:", trainer.state.global_step)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
1,13.290571
2,12.833971
3,12.919748
4,12.125947
5,9.593677
6,10.233724
7,9.166623
8,9.271619
9,8.547782
10,8.158440


训练结果: {'train_runtime': 68.7306, 'train_samples_per_second': 1.877, 'train_steps_per_second': 0.48, 'total_flos': 2536217590639104.0, 'train_loss': 8.293173024148652, 'epoch': 3.0}
Optimizer Steps: 33


In [69]:
# 保存 LoRA Adapter

adapter_dir = "./sft-output/gemma4-e2b-lora-sft/final_adapter"

trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("LoRA Adapter 已保存:", adapter_dir)

LoRA Adapter 已保存: ./sft-output/gemma4-e2b-lora-sft/final_adapter


In [70]:
# 保存训练日志

import pandas as pd

train_logs = [
    log
    for log in trainer.state.log_history
    if "loss" in log
]

df_logs = pd.DataFrame(train_logs)

display(df_logs.head())

df_logs.to_csv(
    "./sft-output/gemma4-e2b-lora-sft/train_logs.csv",
    index=False
)

print("训练日志已保存")

,loss,grad_norm,learning_rate,epoch,step
0,13.290571,4.847286,0.000000,0.093023,1
1,12.833971,5.085133,0.000200,0.186047,2
2,12.919748,4.971262,0.000200,0.279070,3
3,12.125947,5.084663,0.000198,0.372093,4
4,9.593677,4.347894,0.000196,0.465116,5


训练日志已保存


## 10. 模型加载、推理与效果评估

In [71]:
# 加载基础模型

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForMultimodalLM
)

MODEL_ID = "google/gemma-4-E2B-it"

ADAPTER_DIR = (
    "./sft-output/gemma4-e2b-lora-sft/final_adapter"
)

# 加载保存的 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR
)

# 重新加载未经微调的基础模型
base_model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
    low_cpu_mem_usage=True
)

print("基础模型加载成功")
print("模型类型:", type(base_model).__name__)

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

基础模型加载成功
模型类型: Gemma4ForConditionalGeneration


In [72]:
# 加载 LoRA Adapter

from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False
)

model.eval()
model.config.use_cache = True

print("LoRA Adapter 加载成功")
print("模型类型:", type(model).__name__)
print("当前 Adapter:", model.active_adapter)

LoRA Adapter 加载成功
模型类型: PeftModelForCausalLM
当前 Adapter: default


In [73]:
# 验证推理模式下的参数状态

trainable_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("可训练参数数目:", trainable_count)

assert trainable_count == 0

print("模型已经进入推理模式")

可训练参数数目: 0
模型已经进入推理模式


In [77]:
# 模型推理函数

import torch

@torch.inference_mode()
def generate_response(
    model,
    tokenizer,
    messages,
    tools,
    use_adapter=True,
    max_new_tokens=128
):
    model.eval()

    device = next(model.parameters()).device

    # 1. 使用与训练一致的 Chat Template
    encoded = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
        return_dict=True
    )

    # 2. 从 BatchEncoding 中提取 Tensor
    input_ids = encoded["input_ids"].to(device)

    # 有些模板或 Tokenizer 会返回 attention_mask
    attention_mask = encoded.get("attention_mask")

    if attention_mask is None:
        attention_mask = torch.ones_like(input_ids)
    else:
        attention_mask = attention_mask.to(device)

    # 3. 准备生成参数
    generation_kwargs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "pad_token_id": (
            tokenizer.pad_token_id
            if tokenizer.pad_token_id is not None
            else tokenizer.eos_token_id
        )
    }

    # 4. 生成
    if use_adapter:
        output_ids = model.generate(
            **generation_kwargs
        )
    else:
        # 临时禁用 LoRA，使用基础模型
        with model.disable_adapter():
            output_ids = model.generate(
                **generation_kwargs
            )

    # 5. 只提取新生成的 Tokens
    generated_ids = output_ids[
        0, input_ids.shape[1]:
    ]

    # 6. 解码
    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False
    )

    return response

In [78]:
# 测试消息
test_messages = [
    {
        "role": "user",
        "content": (
            "Find the function responsible for loading "
            "session data in this repository. "
            "Inspect the relevant source files before "
            "explaining how it works. "
            "Do not modify any files."
        )
    }
]

In [79]:
# 测试原始模型
base_response = generate_response(
    model=model,
    tokenizer=tokenizer,
    messages=test_messages,
    tools=tools,
    use_adapter=False,
    max_new_tokens=128
)

print("===== 基础模型输出 =====")
print(base_response)

===== 基础模型输出 =====
<|tool_call>call:find{pattern:<|"|>session data loading<|"|>}<tool_call|><|tool_response>


In [80]:
# 测试LoRA 微调模型

sft_response = generate_response(
    model=model,
    tokenizer=tokenizer,
    messages=test_messages,
    tools=tools,
    use_adapter=True,
    max_new_tokens=128
)

print("===== SFT 模型输出 =====")
print(sft_response)

===== SFT 模型输出 =====
<|tool_call>call:find{pattern:<|"|>session data loading<|"|>}<tool_call|><|tool_response>


In [82]:
# 比较基础模型和 SFT 模型的 Loss

import torch

@torch.inference_mode()
def compare_model_loss(model, sample, data_collator):
    model.eval()

    batch = data_collator([sample])

    device = next(model.parameters()).device

    batch = {
        k: v.to(device)
        for k, v in batch.items()
    }

    # 1. 禁用 LoRA，计算基础模型 Loss
    with model.disable_adapter():
        base_loss = model(**batch).loss.item()

    # 2. 启用 LoRA，计算 SFT 模型 Loss
    sft_loss = model(**batch).loss.item()

    return {
        "base_loss": base_loss,
        "sft_loss": sft_loss,
        "loss_difference": base_loss - sft_loss
    }

short_idx = min(
    range(len(train_dataset)),
    key=lambda i: len(train_dataset[i]["input_ids"])
)

result = compare_model_loss(
    model,
    train_dataset[short_idx],
    data_collator
)

print("基础模型 Loss:", result["base_loss"])
print("SFT 模型 Loss:", result["sft_loss"])
print("Loss 改善量:", result["loss_difference"])

基础模型 Loss: 0.47689762711524963
SFT 模型 Loss: 0.6035279035568237
Loss 改善量: -0.1266302764415741


In [83]:
# 比较整个训练集的 Base / SFT Loss

import pandas as pd
import torch

@torch.inference_mode()
def evaluate_base_vs_sft(
    model,
    dataset,
    data_collator
):
    model.eval()

    results = []

    device = next(model.parameters()).device

    for i, sample in enumerate(dataset):
        batch = data_collator([sample])
        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        # 原始模型
        with model.disable_adapter():
            base_loss = model(**batch).loss.item()

        # 微调模型
        sft_loss = model(**batch).loss.item()

        results.append({
            "index": i,
            "seq_len": len(sample["input_ids"]),
            "target_tokens": sum(
                x != -100 for x in sample["labels"]
            ),
            "base_loss": base_loss,
            "sft_loss": sft_loss,
            "improvement": base_loss - sft_loss
        })

    return pd.DataFrame(results)

df_eval = evaluate_base_vs_sft(
    model,
    train_dataset,
    data_collator
)

display(df_eval.head())

print(
    "Base 平均 Loss:",
    df_eval["base_loss"].mean()
)

print(
    "SFT 平均 Loss:",
    df_eval["sft_loss"].mean()
)

print(
    "Loss 改善的样本数量:",
    (df_eval["improvement"] > 0).sum()
)

print(
    "Loss 变差的样本数量:",
    (df_eval["improvement"] < 0).sum()
)

,index,seq_len,target_tokens,base_loss,sft_loss,improvement
0,0,943,38,2.160374,2.127935,0.032439
1,1,1199,86,1.593120,1.676459,-0.083339
2,2,1450,21,0.831659,0.775956,0.055703
3,3,1614,98,4.134823,4.191269,-0.056446
4,4,836,82,4.043510,4.006199,0.037312


Base 平均 Loss: 2.9911847630905553
SFT 平均 Loss: 2.9585674335790233
Loss 改善的样本数量: 29
Loss 变差的样本数量: 14


In [84]:
import numpy as np

weights = df_eval["target_tokens"]

base_weighted_loss = np.average(
    df_eval["base_loss"],
    weights=weights
)

sft_weighted_loss = np.average(
    df_eval["sft_loss"],
    weights=weights
)

print("Base Token 加权 Loss:", base_weighted_loss)
print("SFT Token 加权 Loss:", sft_weighted_loss)
print("整体 Loss 改善:", base_weighted_loss - sft_weighted_loss)

Base Token 加权 Loss: 2.7199895031845585
SFT Token 加权 Loss: 2.7083632626879437
整体 Loss 改善: 0.01162624049661476


## 11. SFT 实验总结

本次实验使用 Gemma 4 E2B 作为基础模型，采用 BF16 + LoRA 进行监督微调。训练数据来自 pi-mono 的单个 Session，经过 Chat Template 渲染、动态上下文裁剪和 Completion-only Loss Masking 后，在 `max_length=2048` 的条件下保留了 43 条训练样本。模型训练 3 个 Epoch，共完成 33 次 Optimizer Step，并成功保存、重新加载 LoRA Adapter。

训练过程中，Training Loss 整体呈下降趋势。微调前后的推理对比显示，两个模型能够生成相同的工具调用。进一步在全部 43 条训练样本上进行评估，基础模型的 Token 加权 Loss 为 2.7200，SFT 模型为 2.7084，其中 29 条样本的 Loss 得到改善，但整体改善幅度较小。

本次实验验证了从 Agent 轨迹数据处理、Dataset 构建、Loss Masking、LoRA 配置，到模型训练、保存、加载及初步评估的完整 SFT 技术流程。但由于训练数据量少、样本高度相关且缺乏独立验证集，目前无法证明模型泛化能力得到提升。后续将扩展到完整 pi-mono 数据集，并采用按 Session 划分训练集与验证集的方式，开展更完整的 SFT 实验。